Сохранение текущего рабочего каталога в переменную HOME

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

Монтируем Google Drive в Colab, чтобы можно было загружать/сохранять файлы. Конкретно в коде это нужно для подгрузки предобученной модели или для сохранения весов во время обучения; что полезно, т.к. Colab ежедневно дает ограниченное кол-во ресурсов, и если не заметить как они закончатся, то можно потерять все данные, полученные во время сессии; а так будет возможность сохранять чекпоинты на диск

In [ ]:
from google.colab import drive
drive.mount(f'{HOME}/drive')

Проверка доступности GPU

In [ ]:
!nvidia-smi

Создание папки для датасета и загрузка Roboflow

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow -q

from roboflow import Roboflow

API_KEY = "ENTER API KEY"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("orion-msydo").project("ship-detection-yr46o")
version = project.version(4)
dataset = version.download("yolov8-obb")


Установка и загрузка YOLO

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

# Загрузка предобученной от ultralytics
# model = YOLO('yolo11x-obb.pt')

# Загрузка предобученной модели с диска
model= YOLO(f'{HOME}/drive/MyDrive/Orion_YOLO_Training/yolov11x_obb6/weights/last.pt')

# Экспорт из Roboflow; но к сожалению это не работает, так как вообще задеплоить obb модель в проект пока нельзя
#model_path = version.model.weights("yolov8-obb")
#model = YOLO(model_path)

Обновление конфигурационного файла для YOLO с obb



In [ ]:
import yaml

with open(f'{dataset.location}/data.yaml', 'r') as f:
    data = yaml.safe_load(f)
data['train'] = '../train/images'
data['val'] = '../valid/images'
data['test'] = '../test/images'
# data['cls'] = 1.0
# data['dropout'] = 0.4
if 'path' in data:
  del data['path']
with open(f'{dataset.location}/data.yaml', 'w') as f:
    yaml.dump(data, f, sort_keys=False)

Удаление military vessel

In [ ]:
import os
import glob

labels_dir = f'{dataset.location}/train/labels/'
images_dir = f'{dataset.location}/train/images/'

# Получаем список всех файлов разметки
label_files = glob.glob(os.path.join(labels_dir, '*.txt'))

for label_file in label_files:
    with open(label_file, 'r') as f:
        lines = f.readlines()
        # Проверяем, есть ли строки, начинающиеся с '5 '
        if any(line.startswith('5 ') for line in lines):
            # Удаляем файл разметки
            os.remove(label_file)

            # Удаляем соответствующий снимок
            image_file = label_file.replace(labels_dir, images_dir).replace('.txt', '.jpg')
            if os.path.exists(image_file):
                os.remove(image_file)

In [ ]:
# Подсчет изображений
train_images = len(os.listdir('Ship-detection-4/train/images'))
valid_images = len(os.listdir('Ship-detection-4/valid/images'))
test_images = len(os.listdir('Ship-detection-4/test/images'))
total_images = train_images + valid_images + test_images

# Вывод результатов
print(f"Total: {total_images}")
print(f"Train: {train_images} ({train_images / total_images:.2%})")
print(f"Valid: {valid_images} ({valid_images / total_images:.2%})")
print(f"Test: {test_images} ({test_images / total_images:.2%})")

---
Запуск обучения

In [ ]:
%cd {HOME}

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch = 8,
    plots=True,
    project=f'{HOME}/drive/MyDrive/Orion_YOLO_Training', # Путь для сохранения на диск (каждую эпоху last.pl и best.pl)
    name="yolov11x_obb"
)

Вывод графика метрик после обучения (потери, точность и т.д.)

In [ ]:
from IPython.display import display
from PIL import Image

#display(Image.open(f'{HOME}/runs/obb/train/results.png'))

# С диска
display(Image.open(f'{HOME}/drive/MyDrive/Orion_YOLO_Training/yolov8x_obb6/results.png'))

In [ ]:
model.val()

Запуск предсказания на тестовых изображениях

In [ ]:
pre = model.predict(
    source=f'{dataset.location}/test/images',
    imgsz=640,
    save=True,
    name='predict',
    conf=0.25
)


Вывод изображений с предсказаниями




In [ ]:
import glob
from IPython.display import display
from PIL import Image

# Получаем список первых 5 предсказанных изображений
predicted_images = sorted(glob.glob(f'{HOME}/datasets/runs/obb/predict/*.jpg'))[:5]

for img_path in predicted_images:
    display(Image.open(img_path))
